# Step 1: Load the benchmark data and recover the gold entity positions

**Goal of this step:** read the labelled dataset and work out *exactly where* each
sensitive entity sits in every sentence. Later steps (regex, ML models, benchmark)
need those positions to check whether a model found the right thing.

**Why it is needed:** the dataset only gives us two strings per example, the original
sentence and a copy where entities are replaced by `<LABEL>` tags. It does **not** tell us
the character positions. We have to line the two strings up ourselves.

The same code lives in the package under `src/anonymizer/spans.py` and
`src/anonymizer/benchmark.py`. This notebook repeats it so you can read and run it step by step.

## 1. The data

`data/benchmark.csv` is a small file (10 examples) with two columns separated by `;`.

Let's load it with pandas and look at it.

In [1]:
# Import modules
from pathlib import Path
import pandas as pd

# The notebook lives in notebooks/, the data lives in data/ one level up.
DATA_PATH = Path("..") / "data" / "benchmark.csv"

# Read the CSV file
df = pd.read_csv(DATA_PATH, sep=";")

print(f"{len(df)} examples, columns: {list(df.columns)}")
df.head(3)

10 examples, columns: ['text', 'label']


,text,label
0,John Smith works for Apple Inc. as a software ...,<PERSON> works for <ORG>. as a <JOB>. His emai...
1,Paris is the capital of France. It is a popula...,<LOCATION> is the capital of <LOCATION>. It is...
2,Google has acquired YouTube for $1.65 billion....,<ORG> has acquired <ORG> for <AMOUNT>. The dea...


Check one example printed in full to clearly see the relationship between the two columns:

In [2]:
print("TEXT :", df.text[0])
print("LABEL:", df.label[0])

TEXT : John Smith works for Apple Inc. as a software engineer. His email address is john.smith@apple.com. He recently traveled to London for a conference.
LABEL: <PERSON> works for <ORG>. as a <JOB>. His email address is <EMAIL_ADDRESS>. He recently traveled to <LOCATION> for a conference.


## 2. A common language: the `Span`

A **span** is one piece of sensitive text: *where* it is and *what kind* it is.

Example: in `"John Smith works for Apple"`, the name `John Smith` is a span that starts at
character 0, ends at character 10, and has the label `PERSON`.

ToDo: Every part of the project (regex, ML models, benchmark, anonymizer) should produce or consume
spans, so they can all be compared and combined in the same way.

In [ ]:
# Import modules
from dataclasses import dataclass

# Set the 12 kinds of sensitive information to detect
TARGET_LABELS = [
    "PERSON",
    "ORG",
    "JOB",
    "EMAIL_ADDRESS",
    "LOCATION",
    "AMOUNT",
    "DATE_TIME",
    "UNIVERSITY",
    "PHONE_NUMBER",
    "URL",
    "IBAN",
    "SSN",
]


@dataclass
class Span:
    """One detected entity: where it is in the text and what kind it is."""

    start: int  # index of the first character
    end: int  # index just after the last character, so text[start:end] is the entity
    label: str  # one of TARGET_LABELS, e.g. "PERSON"
    source: str = ""  # who found it: "gold" (the answer key, i.e. from the benchmark), "regex", "spacy", ...
    score: float = 1.0  # how confident the finder was (1.0 = certain), 
                        # 1.0 by default when source is regex or model doesn't provide a confidence score

    def text(self, full_text: str) -> str:
        """Return the actual words this span covers inside the full text."""
        return full_text[self.start : self.end]


# Test Span class
sentence = "John Smith works for Apple"
test_span = Span(start=0, end=10, label="PERSON", source="demo")
print(test_span)
print("covers:", repr(test_span.text(sentence)))

Span(start=0, end=10, label='PERSON', source='demo', score=1.0)
covers: 'John Smith'


## 3. Lining up `text` and `label`: `align_labels`

In [6]:
# Import modules
import re

# Finds placeholders like <PERSON> or <EMAIL_ADDRESS> inside the label string.
PLACEHOLDER = re.compile(r"<([A-Z_]+)>")


def align_labels(text: str, label: str) -> list[Span]:
    """
    Work out where each entity sits in the original text.

    Returns the entities as a list of Span objects, in reading order.
    Raises ValueError if the label string does not line up with the text.
    """
    pattern_parts = []
    labels = []

    # re.split with a capturing group gives: [words, LABEL, words, LABEL, ..., words]
    pieces = PLACEHOLDER.split(label)
    for i, piece in enumerate(pieces):
        if i % 2 == 0:
            # Even positions are literal text: it must match character for character.
            # re.escape makes sure symbols like "." or "$" are taken literally.
            pattern_parts.append(re.escape(piece))
        else:
            # Odd positions are label names: capture at least one character here.
            # "+?" means "as little as possible", so the slot stops as soon as the
            # following literal text can match.
            pattern_parts.append("(.+?)")
            labels.append(piece)

    pattern = "".join(pattern_parts)
    match = re.fullmatch(pattern, text, flags=re.DOTALL)
    if match is None:
        raise ValueError(f"Label does not line up with text:\n  text : {text}\n  label: {label}")

    # Group 1 is the first slot, group 2 the second, and so on.
    return [
        Span(start=match.start(i + 1), end=match.end(i + 1), label=lab, source="gold")
        for i, lab in enumerate(labels)
    ]

In [ ]:
# Test getting splits: [words, LABEL, words, LABEL, ..., words]
example_label = "<PERSON> works for <ORG>. as a <JOB>."
for i, piece in enumerate(PLACEHOLDER.split(example_label)):
    kind = "literal text" if i % 2 == 0 else "LABEL"
    print(f"{i}: {kind:<12} {piece!r}")

0: literal text ''
1: LABEL        'PERSON'
2: literal text ' works for '
3: LABEL        'ORG'
4: literal text '. as a '
5: LABEL        'JOB'
6: literal text '.'


### Trying it on one example

In [18]:
text, label = df.text[2], df.label[2]
print(text)
print(label)
print()
for span in align_labels(text, label):
    print(f"  {span.label:<10} {span.start:>3}-{span.end:<3} {span.text(text)!r}")

Google has acquired YouTube for $1.65 billion. The deal was finalized in 2006.
<ORG> has acquired <ORG> for <AMOUNT>. The deal was finalized in <DATE_TIME>.

  ORG          0-6   'Google'
  ORG         20-27  'YouTube'
  AMOUNT      32-45  '$1.65 billion'
  DATE_TIME   73-77  '2006'


## 4. Run it on the whole dataset

`load_gold` is a small convenience: load the file and align every row in one go.
Then we print all recovered entities so they can be checked by eye.

In [19]:
"""Get (text, gold spans) for every row."""
gold_list = [(row.text, align_labels(row.text, row.label)) for row in df.itertuples()]

# print all recovered entities
for i, (text, spans) in enumerate(gold_list, start=1):
    print(f"--- example {i}")
    print(text)
    for span in spans:
        print(f"  {span.label:<14} {span.start:>3}-{span.end:<3} {span.text(text)!r}")

--- example 1
John Smith works for Apple Inc. as a software engineer. His email address is john.smith@apple.com. He recently traveled to London for a conference.
  PERSON           0-10  'John Smith'
  ORG             21-30  'Apple Inc'
  JOB             37-54  'software engineer'
  EMAIL_ADDRESS   77-97  'john.smith@apple.com'
  LOCATION       123-129 'London'
--- example 2
Paris is the capital of France. It is a popular tourist destination known for its museums and art galleries.
  LOCATION         0-5   'Paris'
  LOCATION        24-30  'France'
--- example 3
Google has acquired YouTube for $1.65 billion. The deal was finalized in 2006.
  ORG              0-6   'Google'
  ORG             20-27  'YouTube'
  AMOUNT          32-45  '$1.65 billion'
  DATE_TIME       73-77  '2006'
--- example 4
Barack Obama was the President of the United States from 2009 to 2017. He was born in Honolulu, Hawaii and attended Harvard Law School. His phone number is +1 (202) 555-0199.
  PERSON           0-1

## 5. What the output tells about the data

Everything lines up, but the answer key itself is not perfect. Things to remember when
judging the models later:

- **Trailing period is inconsistent.** Example 1 tags `Apple Inc` (label `<ORG>.`), example 7
  tags `Apple Inc.` (label `<ORG> `). A model that always includes or always drops the period
  will be "wrong" on one of them. ToDo: Should I report a *lenient*
  score in the benchmark so that it tolerates small boundary differences?
- **Some labels are debatable.** The area code `212` is tagged `LOCATION`, and `US` in
  "naturalized US citizen" is tagged `LOCATION`.
- **Some entities are missing.** `The Eiffel Tower` in example 5 is not tagged at all.
- **Some target types never appear.** There is no `IBAN` or `SSN` in the 10 examples. So for the benchmark, I will need to add a few examples.